In [1]:
import pandas as pd

def clean_total_crimes(file_path):
    df = pd.read_excel(file_path, header=None)

    df = df.iloc[3:, 1:5]
    df.columns = ['state', '2021', '2022', '2023']

    df = df[df['state'].notna()]
    df = df[df['state'] != 'Total']
    df = df[~df['state'].astype(str).str.contains(r'\[', na=False)]

    df['state'] = df['state'].astype(str).str.strip()

    df_long = df.melt(
        id_vars='state',
        var_name='year',
        value_name='total_crimes'
    )

    df_long['year'] = df_long['year'].astype(int)
    df_long['total_crimes'] = pd.to_numeric(df_long['total_crimes'], errors='coerce')

    return df_long.reset_index(drop=True)


df_total = clean_total_crimes(r"C:\Users\arka7\Downloads\total_ipc_crime.xlsx")

In [2]:
df_total = df_total.sort_values(by=['state', 'year'])

df_total['pct_change'] = df_total.groupby('state')['total_crimes'].pct_change() * 100

df_pivot = df_total.pivot(index='state', columns='year', values='total_crimes').reset_index()

df_pivot['pct_21_22'] = ((df_pivot[2022] - df_pivot[2021]) / df_pivot[2021]) * 100
df_pivot['pct_22_23'] = ((df_pivot[2023] - df_pivot[2022]) / df_pivot[2022]) * 100
df_pivot['pct_21_23'] = ((df_pivot[2023] - df_pivot[2021]) / df_pivot[2021]) * 100

In [3]:
df_total = df_total.sort_values(by=['state', 'year'])

df_total['pct_change'] = df_total.groupby('state')['total_crimes'].pct_change() * 100

df_pivot = df_total.pivot(index='state', columns='year', values='total_crimes').reset_index()

df_pivot['pct_21_22'] = ((df_pivot[2022] - df_pivot[2021]) / df_pivot[2021]) * 100
df_pivot['pct_22_23'] = ((df_pivot[2023] - df_pivot[2022]) / df_pivot[2022]) * 100
df_pivot['pct_21_23'] = ((df_pivot[2023] - df_pivot[2021]) / df_pivot[2021]) * 100

In [4]:
def clean_crime_file(file_path, year):
    df = pd.read_excel(file_path, header=None)

    df_data = df.iloc[9:, :]

    df_states = df_data[[1]].copy()
    df_states.columns = ['state']
    df_states = df_states[df_states['state'].notna()]
    df_states['state'] = df_states['state'].astype(str).str.strip()

    crime_columns = list(range(2, df.shape[1], 3))
    df_crimes = df_data[crime_columns]

    crime_names = df.iloc[3, crime_columns]
    df_crimes.columns = crime_names

    df_final = pd.concat([
        df_states.reset_index(drop=True),
        df_crimes.reset_index(drop=True)
    ], axis=1)

    df_long = df_final.melt(
        id_vars='state',
        var_name='crime_type',
        value_name='count'
    )

    df_long['count'] = pd.to_numeric(df_long['count'], errors='coerce')
    df_long = df_long.dropna()
    df_long = df_long[df_long['state'] != 'Total']

    df_long['crime_type'] = df_long['crime_type'].astype(str).str.strip()
    df_long['year'] = year

    return df_long

In [5]:
df_2021 = clean_crime_file(r"C:\Users\arka7\Downloads\crime_2021.xlsx", 2021)
df_2022 = clean_crime_file(r"C:\Users\arka7\Downloads\crime_2022.xlsx", 2022)
df_2023 = clean_crime_file(r"C:\Users\arka7\Downloads\crime_head_wise.xlsx", 2023)

df_all = pd.concat([df_2021, df_2022, df_2023]).reset_index(drop=True)

df_all['state'] = df_all['state'].str.strip().str.lower()
df_all['crime_type'] = df_all['crime_type'].str.strip().str.lower()

In [6]:
df_national = df_all.groupby(['crime_type', 'year'])['count'].sum().reset_index()

df_trends = df_national.pivot(index='crime_type', columns='year', values='count').reset_index()

df_trends = df_trends.replace(0, pd.NA)

df_trends['pct_21_22'] = ((df_trends[2022] - df_trends[2021]) / df_trends[2021]) * 100
df_trends['pct_22_23'] = ((df_trends[2023] - df_trends[2022]) / df_trends[2022]) * 100
df_trends['pct_21_23'] = ((df_trends[2023] - df_trends[2021]) / df_trends[2021]) * 100

In [7]:
def clean_women_data(file_path):
    df = pd.read_excel(file_path, header=None)

    df = df.iloc[3:, 1:5]
    df.columns = ['state', '2021', '2022', '2023']

    df = df[df['state'].notna()]
    df = df[df['state'] != 'Total']
    df = df[~df['state'].astype(str).str.contains(r'\[', na=False)]

    df['state'] = df['state'].astype(str).str.strip()

    df_long = df.melt(
        id_vars='state',
        var_name='year',
        value_name='crimes_against_women'
    )

    df_long['year'] = df_long['year'].astype(int)
    df_long['crimes_against_women'] = pd.to_numeric(df_long['crimes_against_women'], errors='coerce')

    return df_long.reset_index(drop=True)


df_women = clean_women_data(r"C:\Users\arka7\Downloads\women_crime.xlsx")

In [10]:
# =========================
# FILTER 2023 DATA
# =========================
df_total_2023 = df_all[df_all['year'] == 2023]
df_total_2023 = df_total_2023.groupby('state')['count'].sum().reset_index()
df_total_2023 = df_total_2023.rename(columns={'count': 'total_crimes'})

df_women_2023 = df_women[df_women['year'] == 2023]

# =========================
# 🔥 NORMALIZE STATES (MOST IMPORTANT FIX)
# =========================
df_total_2023['state'] = df_total_2023['state'].astype(str).str.strip().str.lower()
df_women_2023['state'] = df_women_2023['state'].astype(str).str.strip().str.lower()

# =========================
# HANDLE NAME MISMATCHES
# =========================
state_map = {
    "andaman & nicobar islands": "andaman and nicobar islands",
    "dadra & nagar haveli and daman & diu": "dadra and nagar haveli and daman and diu",
    "delhi": "nct of delhi",
    "odisha": "orissa",
    "jammu & kashmir": "jammu and kashmir"
}

df_total_2023['state'] = df_total_2023['state'].replace(state_map)
df_women_2023['state'] = df_women_2023['state'].replace(state_map)

# =========================
# MERGE (SAFE)
# =========================
df_merged = df_total_2023.merge(df_women_2023, on='state', how='inner')

# =========================
# CALCULATE PERCENTAGE
# =========================
df_merged['women_crime_percentage'] = (
    df_merged['crimes_against_women'] / df_merged['total_crimes']
) * 100

# =========================
# DEBUG (VERY IMPORTANT)
# =========================
print("Total states:", df_total_2023.shape[0])
print("Women states:", df_women_2023.shape[0])
print("Merged states:", df_merged.shape[0])

print(df_merged.head())

# =========================
# SAVE
# =========================
df_merged.to_csv("ncrb_women_analysis_2023.csv", index=False)

Total states: 38
Women states: 39
Merged states: 38
               state  total_crimes  year  crimes_against_women  \
0     andhra pradesh       61033.0  2023                 22418   
1  arunachal pradesh        1569.0  2023                   326   
2              assam       31959.0  2023                 12070   
3              bihar      142239.0  2023                 22952   
4         chandigarh         365.0  2023                   371   

   women_crime_percentage  
0               36.730949  
1               20.777565  
2               37.767139  
3               16.136221  
4              101.643836  


C:\Users\arka7\AppData\Local\Temp\ipykernel_14264\4020067277.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_women_2023['state'] = df_women_2023['state'].astype(str).str.strip().str.lower()
C:\Users\arka7\AppData\Local\Temp\ipykernel_14264\4020067277.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_women_2023['state'] = df_women_2023['state'].replace(state_map)


In [12]:
df_total.to_csv("ncrb_total_trends.csv", index=False)
df_pivot.to_csv("ncrb_yoy_changes.csv", index=False)
df_all.to_csv("ncrb_master_dataset.csv", index=False)
df_trends.to_csv("ncrb_crime_trends.csv", index=False)
df_merged.to_csv("ncrb_women_analysis_2023.csv", index=False)